In [ ]:
# One-cell CSV postprocess (copy identical -> enforce best-so-far UB/LB per row -> recompute gaps when overwritten)
#
# Behavior you asked:
# 1) Make an exact-named copy with suffix (refuse overwrite; print "已有同名文件")
# 2) Scan rows sequentially, maintain:
#       best_ub = min(UB[0..k])
#       best_lb = max(LB[0..k])
# 3) For EACH row k:
#       - If the row's UB or LB is NOT equal to (best_ub, best_lb),
#         then overwrite UB/LB with (best_ub, best_lb) AND recompute Abs. Gap + Rel. Gap for that row.
#       - If UB/LB already equal to best-so-far, leave the row untouched (including original gap strings).
#
# This preserves non-numeric formatting (spaces, trailing spaces, 'inf', percent strings) as CSV fields.

from pathlib import Path
import csv, re, math, shutil

# ===== CONFIG =====
in_csv = "simp_1e3sc10_raw.csv"  # <-- change to your input file name
suffix = "_post"           # output suffix
# ==================

in_path = Path(in_csv)
if not in_path.exists():
    raise FileNotFoundError(f"Input not found: {in_path.resolve()}")

out_path = in_path.with_name(in_path.stem + suffix + in_path.suffix)
if out_path.exists():
    print("已有同名文件：", out_path.name)
    raise SystemExit(0)

# --- helpers ---
def dec_places_from_token(tok: str):
    tok = str(tok)
    if tok.lower().startswith("inf"):
        return None
    if tok.endswith("%"):
        tok = tok[:-1]
    tok = tok.strip()
    if re.fullmatch(r"-?\d+(\.\d+)?", tok):
        return len(tok.split(".")[1]) if "." in tok else 0
    return None

def parse_float_token(tok: str):
    tok = str(tok).strip()
    if tok == "":
        return None
    if tok.lower().startswith("inf"):
        return math.inf
    if tok.endswith("%"):
        tok = tok[:-1]
    try:
        return float(tok)
    except Exception:
        return None

def strip_trailing_zeros(s: str):
    if "." not in s:
        return s
    return s.rstrip("0").rstrip(".")

def fmt_float_like(tok_template: str, value: float, fallback_decimals: int):
    if value is None:
        return tok_template
    if math.isinf(value):
        if str(tok_template).lower().startswith("inf"):
            return str(tok_template)
        return "inf"
    d = dec_places_from_token(tok_template)
    if d is None:
        d = fallback_decimals
    s = f"{value:.{d}f}"
    return strip_trailing_zeros(s)

def fmt_percent_like(tok_template: str, percent_value: float, fallback_decimals: int):
    if percent_value is None:
        return tok_template
    if math.isinf(percent_value):
        if str(tok_template).lower().startswith("inf"):
            return str(tok_template)
        return "inf%"
    d = dec_places_from_token(tok_template)
    if d is None:
        d = fallback_decimals
    s = f"{percent_value:.{d}f}"
    s = strip_trailing_zeros(s)
    return s + "%"

def approx_equal(a: float, b: float, tol=0.0):
    # exact match for inf
    if a is None or b is None:
        return False
    if math.isinf(a) or math.isinf(b):
        return (math.isinf(a) and math.isinf(b))
    return abs(a - b) <= tol

# 1) copy file first
shutil.copyfile(in_path, out_path)

# 2) read copied file, make rows safe-length (skip empty lines)
with out_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    H = len(header)
    rows = []
    for r in reader:
        if not r or (len(r) == 1 and str(r[0]).strip() == ""):
            continue
        if len(r) < H:
            r = r + [""] * (H - len(r))
        elif len(r) > H:
            r = r[:H]
        rows.append(r)

def idx(colname: str):
    try:
        return header.index(colname)
    except ValueError:
        raise ValueError(f"Missing column '{colname}'. Header={header}")

# required cols (your file has these exact names)
lb_i  = idx("LB")
ub_i  = idx("UB")
rel_i = idx("Rel. Gap")
abs_i = idx("Abs. Gap")

# infer fallback decimals by simple mode
def fallback_decimals_for_col(i, is_percent=False, default=6):
    counts = {}
    for r in rows:
        tok = r[i]
        v = parse_float_token(tok)
        d = dec_places_from_token(tok)
        if d is not None and v is not None and math.isfinite(v):
            counts[d] = counts.get(d, 0) + 1
    if not counts:
        return (4 if is_percent else default)
    return max(counts.items(), key=lambda x: x[1])[0]

lb_dec  = fallback_decimals_for_col(lb_i,  is_percent=False, default=8)
ub_dec  = fallback_decimals_for_col(ub_i,  is_percent=False, default=8)
abs_dec = fallback_decimals_for_col(abs_i, is_percent=False, default=8)
rel_dec = fallback_decimals_for_col(rel_i, is_percent=True,  default=4)

best_ub = math.inf
best_lb = -math.inf
patched_rows = 0

for r in rows:
    raw_ub = parse_float_token(r[ub_i])
    raw_lb = parse_float_token(r[lb_i])

    # update bests using RAW values
    if raw_ub is not None:
        best_ub = min(best_ub, raw_ub)
    if raw_lb is not None:
        best_lb = max(best_lb, raw_lb)

    # decide whether this row needs overwriting
    need_overwrite = False
    if raw_ub is not None and not approx_equal(raw_ub, best_ub, tol=0.0):
        need_overwrite = True
    if raw_lb is not None and not approx_equal(raw_lb, best_lb, tol=0.0):
        need_overwrite = True

    if need_overwrite:
        abs_gap = best_ub - best_lb
        rel_ratio = abs_gap / max(abs(best_ub), 1e-12)
        rel_percent = rel_ratio * 100.0

        r[ub_i]  = fmt_float_like(r[ub_i],  best_ub, ub_dec)
        r[lb_i]  = fmt_float_like(r[lb_i],  best_lb, lb_dec)
        r[abs_i] = fmt_float_like(r[abs_i], abs_gap, abs_dec)
        r[rel_i] = fmt_percent_like(r[rel_i], rel_percent, rel_dec)

        patched_rows += 1

# 3) rewrite copied file in-place
with out_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, lineterminator="\n")
    writer.writerow(header)
    writer.writerows(rows)

print("Done.")
print("Output:", out_path.name)
print("Rows overwritten (UB/LB differed from best-so-far):", patched_rows)
